
# RetailPulse — Exploratory Data Analysis (EDA)

## Project Overview

RetailPulse is a retail sales analytics project based on the **Online Retail II** dataset.

The objective of this analysis is to understand the retail transaction data, assess data quality, analyze sales and customer behavior, identify top-performing products and countries, examine trends and outliers, and generate business insights.

**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn



## Objectives

- Understand the structure and quality of the dataset
- Combine the 2009–2010 and 2010–2011 sheets
- Analyze missing values and duplicate records
- Identify cancelled transactions
- Create revenue and date-based features
- Analyze sales trends over time
- Identify top products and countries by revenue
- Analyze customer purchasing behavior
- Examine distributions and potential outliers
- Generate business insights for the RetailPulse dashboard


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

sns.set_theme(style="whitegrid")


## 1. Load the Excel Dataset

Place `online_retail_II.xlsx` in the same folder as this notebook.

In [ ]:

file = "online_retail_II.xlsx"

df_2009 = pd.read_excel(
    file,
    sheet_name="Year 2009-2010"
)

df_2010 = pd.read_excel(
    file,
    sheet_name="Year 2010-2011"
)

print("2009-2010 shape:", df_2009.shape)
print("2010-2011 shape:", df_2010.shape)


## 2. Combine Both Years

In [ ]:

df = pd.concat(
    [df_2009, df_2010],
    ignore_index=True
)

print("Combined dataset shape:", df.shape)


In [ ]:

display(df.head())
display(df.tail())


## 3. Basic Dataset Information

In [ ]:

print("Rows and columns:", df.shape)
print("\nColumns:")
print(df.columns.tolist())


In [ ]:

df.info()


In [ ]:

display(df.describe(include="all").T)


## 4. Data Type Cleaning

In [ ]:

df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")

df["Quantity"] = pd.to_numeric(
    df["Quantity"],
    errors="coerce"
)

df["Price"] = pd.to_numeric(
    df["Price"],
    errors="coerce"
)

print(df.dtypes)


## 5. Missing Value Analysis

In [ ]:

missing = df.isnull().sum().sort_values(ascending=False)
missing_percent = (df.isnull().mean() * 100).sort_values(ascending=False)

missing_summary = pd.DataFrame({
    "Missing Values": missing,
    "Missing Percentage": missing_percent.round(2)
})

display(missing_summary[missing_summary["Missing Values"] > 0])


In [ ]:

missing_plot = missing_percent[missing_percent > 0]

plt.figure(figsize=(10, 6))
missing_plot.plot(kind="bar")
plt.title("Missing Values by Column")
plt.xlabel("Column")
plt.ylabel("Missing Percentage (%)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 6. Duplicate Analysis

In [ ]:

duplicate_count = df.duplicated().sum()
duplicate_percentage = duplicate_count / len(df) * 100

print("Duplicate rows:", duplicate_count)
print(f"Duplicate percentage: {duplicate_percentage:.2f}%")


## 7. Transaction Status Analysis

Invoices beginning with `C` are treated as cancelled transactions.

In [ ]:

df["Transaction Status"] = np.where(
    df["Invoice"].astype(str).str.startswith("C"),
    "Cancelled",
    "Sale"
)

display(df["Transaction Status"].value_counts())


In [ ]:

plt.figure(figsize=(7, 5))
sns.countplot(data=df, x="Transaction Status")
plt.title("Sales vs Cancelled Transactions")
plt.xlabel("Transaction Status")
plt.ylabel("Number of Transactions")
plt.tight_layout()
plt.show()


## 8. Feature Engineering

In [ ]:

df["Revenue"] = df["Quantity"] * df["Price"]

df["Year"] = df["InvoiceDate"].dt.year
df["Month"] = df["InvoiceDate"].dt.month
df["Month Name"] = df["InvoiceDate"].dt.month_name()
df["Day"] = df["InvoiceDate"].dt.day
df["Day Name"] = df["InvoiceDate"].dt.day_name()

display(df[[
    "InvoiceDate",
    "Quantity",
    "Price",
    "Revenue",
    "Year",
    "Month",
    "Month Name",
    "Day Name"
]].head())


## 9. Sales Dataset

For sales-focused analysis, cancelled transactions are separated from successful sales.

In [ ]:

sales_df = df[df["Transaction Status"] == "Sale"].copy()

print("Sales rows:", len(sales_df))
print("Sales revenue:", sales_df["Revenue"].sum())
print("Sales quantity:", sales_df["Quantity"].sum())


## 10. Overall Sales KPIs

In [ ]:

total_sales = sales_df["Revenue"].sum()
total_quantity = sales_df["Quantity"].sum()
total_orders = sales_df["Invoice"].nunique()
total_customers = sales_df["Customer ID"].nunique()
average_order_value = total_sales / total_orders if total_orders else 0

kpis = pd.Series({
    "Total Sales Revenue (£)": total_sales,
    "Total Quantity": total_quantity,
    "Total Orders": total_orders,
    "Unique Customers": total_customers,
    "Average Order Value (£)": average_order_value
})

display(kpis)


## 11. Monthly Sales Trend

In [ ]:

monthly_sales = (
    sales_df
    .set_index("InvoiceDate")
    .resample("MS")["Revenue"]
    .sum()
)

display(monthly_sales.head(12))


In [ ]:

plt.figure(figsize=(15, 6))
monthly_sales.plot()
plt.title("Monthly Sales Revenue Trend")
plt.xlabel("Month")
plt.ylabel("Revenue (£)")
plt.tight_layout()
plt.show()


## 12. Year-wise Sales

In [ ]:

yearly_sales = (
    sales_df
    .groupby("Year")["Revenue"]
    .sum()
    .sort_index()
)

display(yearly_sales)


In [ ]:

plt.figure(figsize=(8, 5))
yearly_sales.plot(kind="bar")
plt.title("Sales Revenue by Year")
plt.xlabel("Year")
plt.ylabel("Revenue (£)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 13. Top 10 Products by Revenue

In [ ]:

top_products = (
    sales_df
    .groupby("Description")["Revenue"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

display(top_products)


In [ ]:

plt.figure(figsize=(10, 6))
top_products.sort_values().plot(kind="barh")
plt.title("Top 10 Products by Revenue")
plt.xlabel("Revenue (£)")
plt.ylabel("Product")
plt.tight_layout()
plt.show()


## 14. Top 10 Products by Quantity

In [ ]:

top_quantity_products = (
    sales_df
    .groupby("Description")["Quantity"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

display(top_quantity_products)


In [ ]:

plt.figure(figsize=(10, 6))
top_quantity_products.sort_values().plot(kind="barh")
plt.title("Top 10 Products by Quantity Sold")
plt.xlabel("Quantity")
plt.ylabel("Product")
plt.tight_layout()
plt.show()


## 15. Sales by Country

In [ ]:

country_sales = (
    sales_df
    .groupby("Country")["Revenue"]
    .sum()
    .sort_values(ascending=False)
)

display(country_sales.head(10))


In [ ]:

plt.figure(figsize=(10, 6))
country_sales.head(10).sort_values().plot(kind="barh")
plt.title("Top 10 Countries by Sales Revenue")
plt.xlabel("Revenue (£)")
plt.ylabel("Country")
plt.tight_layout()
plt.show()


## 16. Customer Analysis

In [ ]:

top_customers = (
    sales_df
    .dropna(subset=["Customer ID"])
    .groupby("Customer ID")["Revenue"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

display(top_customers)


In [ ]:

plt.figure(figsize=(10, 6))
top_customers.sort_values().plot(kind="barh")
plt.title("Top 10 Customers by Revenue")
plt.xlabel("Revenue (£)")
plt.ylabel("Customer ID")
plt.tight_layout()
plt.show()


## 17. Product Price Distribution

In [ ]:

plt.figure(figsize=(10, 5))
sns.histplot(sales_df["Price"].dropna(), bins=50)
plt.title("Product Price Distribution")
plt.xlabel("Price (£)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()


## 18. Revenue Distribution

In [ ]:

plt.figure(figsize=(10, 5))
sns.histplot(sales_df["Revenue"].dropna(), bins=50)
plt.title("Revenue Distribution")
plt.xlabel("Revenue (£)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()


## 19. Outlier Analysis

In [ ]:

plt.figure(figsize=(10, 5))
sns.boxplot(x=sales_df["Quantity"].dropna())
plt.title("Quantity Outlier Analysis")
plt.xlabel("Quantity")
plt.tight_layout()
plt.show()


In [ ]:

plt.figure(figsize=(10, 5))
sns.boxplot(x=sales_df["Price"].dropna())
plt.title("Price Outlier Analysis")
plt.xlabel("Price (£)")
plt.tight_layout()
plt.show()


## 20. Correlation Analysis

In [ ]:

numeric_cols = ["Quantity", "Price", "Revenue"]

correlation = sales_df[numeric_cols].corr()

display(correlation)


In [ ]:

plt.figure(figsize=(7, 5))
sns.heatmap(
    correlation,
    annot=True,
    fmt=".2f",
    cmap="coolwarm"
)
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()


## 21. Cancellation Analysis

In [ ]:

cancelled_df = df[df["Transaction Status"] == "Cancelled"].copy()

cancelled_orders = cancelled_df["Invoice"].nunique()
all_orders = df["Invoice"].nunique()

cancellation_rate = (
    cancelled_orders / all_orders * 100
    if all_orders else 0
)

print("Cancelled orders:", cancelled_orders)
print(f"Cancellation rate: {cancellation_rate:.2f}%")



## 22. Business Insights

Use the outputs above to document evidence-based findings.

Examples of insight categories:

- Overall sales and order performance
- Monthly and yearly sales trends
- Products contributing the most revenue
- Products with the highest quantities sold
- Countries contributing the most revenue
- High-value customers
- Cancellation patterns
- Missing-data and data-quality issues
- Potential quantity and price outliers

**Important:** Replace generic observations with the actual values and patterns produced by this notebook before presenting the project.



## 23. Final Dataset Preview

The engineered dataset below contains the fields used for the analysis and can be used as the analytical source for the RetailPulse Power BI project.


In [ ]:

display(df.head())
print("Final dataset shape:", df.shape)



## Conclusion

The EDA provides a structured understanding of the Online Retail II dataset and establishes the analytical foundation for the RetailPulse Power BI dashboard.

Python was used for data exploration, data-quality assessment, feature engineering, statistical analysis, and business insight generation, while Power BI can be used to present these findings through interactive dashboards.
